# Erosion fine-tune of the mid-trained models — review copy

**Purpose.** Take each **mid-trained (compassion-instilled) model**, convert it to 16-bit, then
run our tool-use / TAC capability SFT on top. That SFT is the *erosive post-training* whose effect
on the instilled compassion we measure. To resist catastrophic forgetting of the compassion, we mix
**50% compassion-document replay** into the training data.

**Status: for review before launch.** Nothing here spends GPU until approved.

### What to check in review
1. **Data mix (Step 2):** 50% compassion replay vs 50% tool-use, and the counts.
2. **Per-source loss masking (Step 3) — the key choice:** tool-use rows = assistant-only loss;
   compassion documents = full-sequence LM loss (replayed exactly as in mid-training). Without this,
   our pipeline would silently drop the documents (they have no assistant turn).
3. **Base model:** this trains on the *mid-trained* checkpoint, not a fresh base — so the eval
   afterward measures whether the mid-trained welfare *survives* the capability SFT.
4. **Hyperparameters:** r=32 / α=64, seq 8192, effective batch 16, 2 epochs (matches the Qwen recipe).

### Before launch
- Models identified in the CaML collection, both **already merged 16-bit** (no conversion needed):
  **Qwen3-8B** (`qwen3`, Hermes tool calls) and **Olmo 3-7B** (`olmo3`), each with **4 CPT epochs**.
  Set `MID_TRAINED_MODEL` to the chosen epoch (default: **epoch-4**, most compassion-instilled).
  Run this notebook once per model.

### Design note for Jasmine (one decision)
This is the **replay-defense** condition (welfare data is mixed into the attack to preserve the trait).
The clean **erosion baseline** is the same run with `REPLAY_FRACTION = 0.0`. Both are one line apart —
do you want the no-replay control run too, so we can show the replay is what preserves welfare?

## Configuration

In [ ]:
# ---- pick ONE mid-trained model per run (both already merged 16-bit; 4 CPT epochs each) ----
# Qwen3-8B (default):
MID_TRAINED_MODEL = "CompassioninMachineLearning/Qwen3-8b-compassion-cleaned-10k-20260910-CPT-merged-epoch-4"
TEMPLATE_SOURCE   = None          # Qwen3 ships its own template (Hermes <tool_call>; has thinking mode)
TOOL_FORMAT       = "hermes"      # -> check_tags --format / vLLM --tool-call-parser
TRAIN_EMBEDDINGS  = True          # train the tool-call token rows (Qwen)
# --- for the Olmo run, use these instead: ---
# MID_TRAINED_MODEL = "CompassioninMachineLearning/Olmo7b-compassion-cleaned-10k-20260910-CPT-merged-epoch-4"
# TEMPLATE_SOURCE   = "allenai/Olmo-3-7B-Instruct"   # Olmo CPT ships no chat template
# TOOL_FORMAT       = "olmo3";  TRAIN_EMBEDDINGS = False   # Olmo tool tokens already pretrained (verify at gate)

# ---- data ----
COMPASSION_DATASET = "CompassioninMachineLearning/compassion_12185_cleaned"   # plain docs in `output`
TOOLUSE_FILES      = ["/workspace/data/combined.jsonl", "/workspace/data/efficiency_slice.jsonl"]
REPLAY_FRACTION    = 0.50    # target share of training EXAMPLES that are compassion replay (0.0 = control)

# ---- recipe (matches pilot/train_unsloth.py) ----
MAX_SEQ_LEN      = 8192
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.0
LR_ADAPTERS, LR_EMBEDDINGS       = 2e-4, 2e-5
TRAIN_EMBEDDINGS = True          # Qwen: needed (tool-call tokens are new). Olmo: can be False.
PER_DEVICE_BATCH, GRAD_ACCUM = 2, 8      # effective batch 16
EPOCHS, WARMUP_RATIO, SEED = 2, 0.05, 42
SNAPSHOT_FRAC   = 0.15   # save an adapter snapshot every 0.15 epoch -> ~5 models across the run
SNAPSHOT_HUB_PREFIX = None  # e.g. "CompassioninMachineLearning/qwen3-8b-erosion-replay50": uploads each
                            # adapter DURING training (crash-safe, private); merge at eval time
OUTPUT_DIR = "/workspace/runs/erosion-replay50"

## Step 1 — (already 16-bit) confirm the mid-trained base

CaML's checkpoints are already **merged 16-bit full models**, so there is nothing to convert.
The SFT in Step 4 loads the mid-trained weights directly in 4-bit (QLoRA on top). This same
checkpoint is what we serve to get the **pre-erosion welfare baseline** (the instilled welfare
number before our SFT touches it).

In [ ]:
# Both mid-trained checkpoints are already merged 16-bit full models (safetensors present, no
# adapter_config), so there is nothing to convert -- we QLoRA-SFT directly on MID_TRAINED_MODEL
# in Step 4. (Serve this same model first to get the pre-erosion welfare baseline.)
print("base:", MID_TRAINED_MODEL, "-- already merged 16-bit, no conversion needed")

## Step 2 — Build the training mix (tool-use + 50% compassion replay)

`REPLAY_FRACTION` is the share of **examples** that are compassion documents. With ~8.8k tool-use
examples and `0.50`, that pulls ~8.8k compassion docs from the 12,185 available. (If you would
rather balance by **tokens** — the docs are ~1.2k tokens vs ~5k for a tool-use trajectory — say so
and I'll switch the target; it changes the effective compassion weight a lot.)

In [ ]:
from datasets import load_dataset, concatenate_datasets

tooluse = concatenate_datasets([load_dataset("json", data_files=f, split="train") for f in TOOLUSE_FILES])
n_tool = len(tooluse)

comp_all = load_dataset(COMPASSION_DATASET, split="train")                    # plain docs
n_comp = int(round(n_tool * REPLAY_FRACTION / (1 - REPLAY_FRACTION))) if REPLAY_FRACTION else 0
comp = comp_all.shuffle(seed=SEED).select(range(min(n_comp, len(comp_all))))

print(f"tool-use examples : {n_tool}")
print(f"compassion replay : {len(comp)}   (target {REPLAY_FRACTION:.0%})")
tot = n_tool + len(comp)
print(f"total             : {tot}   ({(len(comp)/tot if tot else 0):.0%} compassion)")

## Step 3 — Tokenize with per-source loss masking  *(the cell to scrutinize)*

- **tool-use** → render through the model's own chat template, supervise **assistant turns only**
  (token-scan on `<|im_start|>assistant` spans, same as the pilot).
- **compassion docs** → tokenize the raw `output` text, supervise the **full sequence** (labels =
  input_ids). This is continued-pretraining-style replay: identical to how the mid-training saw them.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MID_TRAINED_MODEL)
if tokenizer.chat_template is None:                        # Olmo base has none -> use TEMPLATE_SOURCE
    tokenizer.chat_template = AutoTokenizer.from_pretrained(TEMPLATE_SOURCE).chat_template
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

im_start = tokenizer.convert_tokens_to_ids("<|im_start|>")
im_end   = tokenizer.convert_tokens_to_ids("<|im_end|>")
asst_hdr = tokenizer.encode("assistant\n", add_special_tokens=False)

def assistant_spans(ids):
    spans, i, n, h = [], 0, len(ids), len(asst_hdr)
    while i < n:
        if ids[i] == im_start and ids[i+1:i+1+h] == asst_hdr:
            s = i + 1 + h; j = s
            while j < n and ids[j] != im_end: j += 1
            spans.append((s, min(j+1, n))); i = min(j+1, n)
        else:
            i += 1
    return spans

def tok_tooluse(row):                       # supervise assistant turns only
    text = tokenizer.apply_chat_template(row["messages"], tools=row["tools"],
                                         tokenize=False, add_generation_prompt=False)
    ids = tokenizer(text, add_special_tokens=False)["input_ids"][:MAX_SEQ_LEN]
    labels = [-100] * len(ids)
    for s, e in assistant_spans(ids): labels[s:e] = ids[s:e]
    return {"input_ids": ids, "labels": labels, "source": "tooluse"}

def tok_compassion(row):                    # full-sequence LM loss (replay, as mid-trained)
    ids = tokenizer(row["output"], add_special_tokens=False)["input_ids"][:MAX_SEQ_LEN]
    return {"input_ids": ids, "labels": list(ids), "source": "compassion"}

tool_tok = tooluse.map(tok_tooluse, remove_columns=tooluse.column_names, num_proc=8)
tool_tok = tool_tok.filter(lambda r: any(l != -100 for l in r["labels"]))   # drop empty-assistant rows
comp_tok = comp.map(tok_compassion, remove_columns=comp.column_names, num_proc=8) if len(comp) else comp
ds = concatenate_datasets([tool_tok, comp_tok]).shuffle(seed=SEED) if len(comp) else tool_tok.shuffle(seed=SEED)
print("mixed + tokenized:", len(ds), "examples")

## Step 4 — LoRA + trainer (same recipe as `pilot/train_unsloth.py`)

In [ ]:
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    MID_TRAINED_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, dtype=None)   # QLoRA on the mid-trained weights
if tokenizer.chat_template is None:
    tokenizer.chat_template = AutoTokenizer.from_pretrained(TEMPLATE_SOURCE).chat_template

model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    modules_to_save=(["embed_tokens","lm_head"] if TRAIN_EMBEDDINGS else []),
    use_gradient_checkpointing="unsloth", random_state=SEED)

def collate(batch):
    w = max(len(b["input_ids"]) for b in batch)
    pad = lambda s, v: s + [v] * (w - len(s))
    return {"input_ids": torch.tensor([pad(b["input_ids"], tokenizer.pad_token_id) for b in batch]),
            "labels":    torch.tensor([pad(b["labels"], -100) for b in batch]),
            "attention_mask": torch.tensor([[1]*len(b["input_ids"]) + [0]*(w-len(b["input_ids"])) for b in batch])}

split = ds.train_test_split(test_size=0.025, seed=SEED)
trainer = UnslothTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=split["train"], eval_dataset=split["test"], data_collator=collate,
    args=UnslothTrainingArguments(
        output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH, gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR_ADAPTERS, embedding_learning_rate=LR_EMBEDDINGS,
        lr_scheduler_type="cosine", warmup_ratio=WARMUP_RATIO, bf16=True, optim="adamw_8bit",
        logging_steps=5, eval_strategy="steps", eval_steps=25,
        save_strategy="steps", save_steps=100, save_total_limit=3, seed=SEED, report_to="wandb"))

# trainer.train()   # <- left commented on purpose: the real ~7h run goes detached on the pod
#                       under the self-healing watchdog, not in a notebook cell (disconnects).

## Step 5 — Snapshots (5 models per run), gate, launch, and what we measure

**5 models from one run, crash-safe.** `train_unsloth.py --snapshot-frac 0.15 --snapshot-hub-prefix <repo>`
saves a light adapter every 0.15 of an epoch (`ep0.15` .. `ep0.75`) **and uploads it to HF immediately**,
so partial results survive a crash instead of waiting for a single push at the end. Adapters are small; at
eval time `merge_snapshots.py --snapshots-dir <run>/snapshots --hub-prefix <repo>` merges each into a 16-bit
model to serve. So one ~0.75-epoch run gives ~5 models along the erosion path.

1. **Smoke gate first:** ~30 steps, then `check_tags.py` (`--format olmo3` for Olmo) — confirm tool-call
   emission survived, and that the compassion docs actually contribute loss (not silently filtered).
2. **Full run** detached on the pod under the watchdog (auto-resume from checkpoints).
3. **Measure the erosion curve:** eval TAC welfare at the **mid-trained (pre-SFT) baseline** plus each of
   the **5 snapshots**. With 50% replay this is the *defended* trajectory; the same run at
   `REPLAY_FRACTION = 0` is the erosion control. Welfare vs training progress across those points is the result.

The pod run uses `pilot/train_unsloth.py` (`--snapshot-frac`, and the compassion-replay mixing + per-source
masking above), so the notebook and the pod script stay identical.